# Stress capital and risk margin
This is an invented 10,000-policy Term demonstration, not a regulatory opinion. Install with the ui extra and dev group, then run every cell. The generated stress set retains explicit stress sizes, correlation and cost-of-capital settings.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import plotly.graph_objects as go
import polars as pl

from avk.api import check, run
from avk.demo.generate import generate

folder = Path(mkdtemp(prefix="avk-notebook-"))
generate(folder, "TERM", 10000, 209)
print(f"Invented inputs and results: {folder}")


def execute(mode):
    config = folder / f"{mode}.json"
    check(config)
    return run(config)


def aggregate(result):
    path = next(p for p in result.files if p.name.endswith("_aggregate_output.csv"))
    return pl.read_csv(path)

Each stress revalues the same book; the selected aggregation method combines risks using the supplied correlation matrix. The capital output is separate from the factor-based solvency columns in valuation. Step 1 is the reported date row.


In [ ]:
capital = aggregate(execute("capital"))
row = capital.filter(pl.col("Projection_Step") == 1)
assert row.height == 1
columns = [
    "capital_mortality",
    "capital_lapse",
    "capital_expense",
    "capital_life catastrophe",
    "total_capital",
]
print(row.select(columns + ["pv_cost_of_capital"]))
fig = go.Figure(go.Bar(x=columns, y=[row[c][0] for c in columns]))
fig.update_layout(title="Stress capital at step 1", yaxis_title="Currency units")
fig.show()

The monthly `cost_of_capital` is the cost of holding future capital; `pv_cost_of_capital` discounts the remaining cost stream on the configured curve. Interpret this as the example's risk margin basis, distinct from an IFRS 17 risk adjustment. The measurement notebook inputs choose a ratio RA, not this cost-of-capital method.


In [ ]:
assert capital["total_capital"].min() >= 0
fig = go.Figure()
for column in ("total_capital", "pv_cost_of_capital"):
    fig.add_scatter(
        x=capital["Projection_Step"].to_list(), y=capital[column].to_list(), name=column
    )
fig.update_layout(
    title="Capital run-off and present value of holding costs",
    xaxis_title="Projection step",
    yaxis_title="Currency units",
)
fig.show()